# KDHS 2022 — Contraceptive Use Prediction
## Data Cleaning, EDA, Modelling & Deployment
### The Insight Architects Group

---

**Dataset:** Kenya Demographic and Health Survey (KDHS) 2022 — Women's Questionnaire  
**Respondents:** 32,156 women aged 15–49 across all 47 counties  
**Target Variable:** `contraceptive_use` — Type of contraceptive method currently used  
**Methodology:** CRISP-DM  
**Tools:** Python · scikit-learn · LightGBM · SHAP · Flask

---

## 1. Business Understanding

### 1.1 Business Overview

Kenya has made significant progress in family planning over the past three decades, yet contraceptive uptake remains deeply unequal across the country's 47 counties. The 2022 Kenya Demographic and Health Survey (KDHS), a nationally representative dataset of 32,156 women aged 15–49, captures this inequality in unprecedented detail — recording sociodemographic variables such as education, wealth, residence, marital status, and county of residence alongside contraceptive use patterns. Since manual analysis of such a large, multi-variable dataset is impractical for health planners, machine learning can be used to automatically identify the sociodemographic profiles of women most at risk of contraceptive non-use.

This project develops a contraceptive use prediction model using the 2022 KDHS dataset to help the Ministry of Health and family planning partners identify underserved population segments and deploy targeted interventions efficiently and at scale.

### 1.2 Problem Statement

Kenya's modern contraceptive prevalence rate among married women is approximately 60% nationally, but this average conceals major geographic and socioeconomic inequalities. In arid and semi-arid counties such as Turkana, West Pokot, and Mandera, contraceptive uptake falls 20–30 percentage points below the national average, while more urbanised counties like Nairobi and Kiambu report markedly higher rates. Without a data-driven segmentation of at-risk populations, health interventions are distributed uniformly rather than targeted where they are needed most.

**Key challenges include:**

- **Class imbalance** — modern method users (38%) significantly outnumber folkloric and traditional method users (4% combined)
- **High dimensionality** — 5,925 variables in the raw survey requiring careful feature selection down to the most relevant predictors
- **Multicollinearity** — closely related variables such as age, children ever born, and marital status overlap in predictive power
- **Categorical encoding** — ordinal variables like wealth index and education level require careful treatment to preserve their natural ordering

**Objective:** Build a model that accurately predicts contraceptive use among Kenyan women aged 15–49 using sociodemographic features, and identify which factors most strongly drive or prevent modern contraceptive uptake.

### 1.3 Stakeholders

| Stakeholder | How They Use This Model |
|---|---|
| Kenya Ministry of Health — Division of Reproductive Health | Identify counties and demographic groups with highest predicted non-use; allocate field officers and resources accordingly |
| County Health Management Teams (all 47 counties) | Generate county-specific risk profiles to guide local family planning outreach programmes |
| UNFPA Kenya Country Office | Target funding and technical assistance toward highest-risk population segments |
| USAID Kenya | Use model outputs to evaluate programme effectiveness and redirect investments to underserved regions |
| FP2030 Programme | Monitor progress toward family planning targets by tracking predicted non-use rates over time |
| Community Health Promoters (CHPs) | Use county-level risk scores to prioritise household visits in high-risk clusters |

### 1.4 Success Metrics

The model will be considered successful if:

- **Accuracy ≥ 78%** on the held-out test set
- **ROC-AUC ≥ 0.82**, measuring the model's ability to distinguish users from non-users
- **Macro F1-Score ≥ 0.75**, ensuring minority classes are not neglected
- **Balanced Precision and Recall**, especially for non-use prediction — failing to identify at-risk women has real public health consequences
- **SHAP explanations identify at least 5 key features** that policymakers can directly act on
- The model generalises to unseen data via stratified 5-fold cross-validation
- The model improves over a simple logistic regression baseline

### 1.5 Key Business Questions

1. What is the overall distribution of contraceptive use types (modern, traditional, folkloric, none) among Kenyan women aged 15–49?
2. Which counties have the highest predicted rates of contraceptive non-use, and what sociodemographic profiles characterise these women?
3. How do education level and wealth index jointly influence the likelihood of modern contraceptive use?
4. Does urban or rural residence remain a significant predictor of contraceptive use when controlling for wealth and education?
5. Which machine learning model best predicts contraceptive use, and which sociodemographic features are the strongest predictors according to SHAP analysis?

---

## Notebook Structure

| # | Section | CRISP-DM Phase |
|---|---|---|
| 1 | Setup & Imports | — |
| 2 | Data Loading & Initial Inspection | Data Understanding |
| 3 | Pre-Cleaning Audit | Data Understanding |
| 4 | Data Cleaning Pipeline | Data Preparation |
| 5 | Post-Cleaning Validation | Data Preparation |
| 6 | Before vs. After Comparison | Data Preparation |
| 7 | EDA — Univariate | Data Understanding |
| 8 | EDA — Bivariate | Data Understanding |
| 9 | EDA — Multivariate | Data Understanding |
| 10 | Key EDA Findings & Modelling Implications | Data Understanding |
| 11 | Feature Engineering | Data Preparation |
| 12 | Train / Test Split & Class Imbalance Handling | Data Preparation |
| 13 | Baseline Model — Logistic Regression | Modelling |
| 14 | Model 2 — Random Forest | Modelling |
| 15 | Model 3 — LightGBM (Tuned) | Modelling |
| 16 | Model Comparison & Selection | Evaluation |
| 17 | SHAP Explainability | Evaluation |
| 18 | Business Recommendations | Evaluation |
| 19 | Deployment — Flask API | Deployment |

## 2 Loading Dataset

In [14]:

import pandas as pd
df = pd.read_csv("KDHS_2022_women.csv")
df.head()

c:\Users\Admin\Nanjala\envs\learn-env\lib\site-packages\IPython\core\interactiveshell.py:3145: DtypeWarning: Columns (5158) have mixed types.Specify dtype option on import or set low_memory=False.
  has_raised = await self.run_ast_nodes(code_ast.body, cell_name,


,caseid,v000,v001,v002,v003,v004,v005,v006,v007,v008,...,s631l_3,s631l_4,s631l_5,s631l_6,s631m_1,s631m_2,s631m_3,s631m_4,s631m_5,s631m_6
0,1 4 2,KE8,1,4,2,1,1296049,4,2022,1468,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1 7 2,KE8,1,7,2,1,1296049,4,2022,1468,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1 10 1,KE8,1,10,1,1,1296049,4,2022,1468,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1 13 2,KE8,1,13,2,1,1296049,4,2022,1468,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1 20 2,KE8,1,20,2,1,1296049,4,2022,1468,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [15]:
df.shape

(32156, 5925)

In [16]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32156 entries, 0 to 32155
Columns: 5925 entries, caseid to s631m_6
dtypes: float64(5689), int64(232), object(4)
memory usage: 1.4+ GB


In [17]:
df.describe()

,v001,v002,v003,v004,v005,v006,v007,v008,v008a,v009,...,s631l_3,s631l_4,s631l_5,s631l_6,s631m_1,s631m_2,s631m_3,s631m_4,s631m_5,s631m_6
count,32156.000000,32156.000000,32156.000000,32156.000000,3.215600e+04,32156.000000,32156.0,32156.000000,32156.000000,32156.000000,...,37.000000,0.0,0.0,0.0,2991.000000,475.000000,37.000000,0.0,0.0,0.0
mean,858.124953,44.294004,2.405057,858.124953,1.000000e+06,4.252923,2022.0,1468.252923,44674.601878,6.090061,...,0.027027,NaN,NaN,NaN,0.245737,0.233684,0.162162,NaN,NaN,NaN
std,491.377924,36.496684,1.564089,491.377924,9.690516e+05,1.372520,0.0,1.372520,40.479124,3.613582,...,0.164399,NaN,NaN,NaN,0.430595,0.423620,0.373684,NaN,NaN,NaN
min,1.000000,0.000000,1.000000,1.000000,4.141600e+04,2.000000,2022.0,1466.000000,44610.000000,1.000000,...,0.000000,NaN,NaN,NaN,0.000000,0.000000,0.000000,NaN,NaN,NaN
25%,424.000000,20.000000,2.000000,424.000000,4.100380e+05,3.000000,2022.0,1467.000000,44638.000000,3.000000,...,0.000000,NaN,NaN,NaN,0.000000,0.000000,0.000000,NaN,NaN,NaN
50%,881.000000,40.000000,2.000000,881.000000,8.303350e+05,4.000000,2022.0,1468.000000,44672.000000,6.000000,...,0.000000,NaN,NaN,NaN,0.000000,0.000000,0.000000,NaN,NaN,NaN
75%,1290.250000,64.000000,3.000000,1290.250000,1.221371e+06,5.000000,2022.0,1469.000000,44709.000000,9.000000,...,0.000000,NaN,NaN,NaN,0.000000,0.000000,0.000000,NaN,NaN,NaN
max,1692.000000,2016.000000,22.000000,1692.000000,9.823301e+06,7.000000,2022.0,1471.000000,44770.000000,12.000000,...,1.000000,NaN,NaN,NaN,1.000000,1.000000,1.000000,NaN,NaN,NaN


In [19]:
cols_needed = ['v012','v013','v024','v025','v106','v190','v501','v502',
               'v201','v218','v228','v714','v136','v130','v151',
               'v701','v212','v302a','v313']

In [20]:
df = pd.read_csv("KDHS_2022_women.csv",
                 usecols=cols_needed, low_memory=False)

In [21]:
df.rename(columns={
    'v012': 'age',
    'v013': 'age_group',
    'v024': 'county',
    'v025': 'residence_type',
    'v106': 'education_level',
    'v190': 'wealth_index',
    'v501': 'marital_status',
    'v502': 'union_status',
    'v201': 'children_ever_born',
    'v218': 'living_children',
    'v228': 'pregnancy_loss',
    'v714': 'currently_working',
    'v136': 'household_size',
    'v130': 'religion',
    'v151': 'household_head_sex',
    'v701': 'partner_education',
    'v212': 'age_first_birth',
    'v302a': 'ever_used_contraceptive',
    'v313': 'contraceptive_use'   #TARGET
}, inplace=True)

## 2.1 Data Cleaning/Preparation

In [22]:
#check missing values
df.isnull().sum()

age                            0
age_group                      0
county                         0
residence_type                 0
education_level                0
religion                       0
household_size                 0
household_head_sex             0
wealth_index                   0
children_ever_born             0
age_first_birth             8813
living_children                0
pregnancy_loss                 0
ever_used_contraceptive        0
contraceptive_use              0
marital_status                 0
union_status                   0
partner_education          14039
currently_working              0
dtype: int64

In [23]:
#check duplicates
df.duplicated().sum()

274

In [24]:
#understand variables
df.columns.tolist()

['age',
 'age_group',
 'county',
 'residence_type',
 'education_level',
 'religion',
 'household_size',
 'household_head_sex',
 'wealth_index',
 'children_ever_born',
 'age_first_birth',
 'living_children',
 'pregnancy_loss',
 'ever_used_contraceptive',
 'contraceptive_use',
 'marital_status',
 'union_status',
 'partner_education',
 'currently_working']

## 3.Exploratory Data Analysis (EDA)